In [ ]:
!pip install mlflow

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import load_iris
import warnings

In [2]:
warnings.filterwarnings("ignore") # Para ignorar warnings de convergencia de sklearn

In [3]:
# 1. Cargar el dataset Iris
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

In [4]:
# 2. Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
import os

In [6]:
# 3. Definir parámetros del modelo
# Usaremos una variable de entorno para simular un parámetro configurable
# En un entorno real, esto podría venir de un archivo de configuración o CLI
alpha = float(os.environ.get("ALPHA", 0.1)) # Ejemplo de parámetro configurable
l1_ratio = float(os.environ.get("L1_RATIO", 0.5)) # Ejemplo de parámetro configurable

In [7]:
# Apuntar al servidor MLflow
experiment_name = "Experimento_Proxy_Final"
mlflow.set_tracking_uri("http://192.168.1.156:5000")

In [8]:
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(
        name=experiment_name,
        artifact_location="mlflow-artifacts:/" 
    )

In [9]:
# Crear o seleccionar un experimento
# mlflow.set_experiment("Pruebas3")
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1772816600449, experiment_id='9', last_update_time=1772816600449, lifecycle_stage='active', name='Experimento_Proxy_Final', tags={}>

In [10]:
# 4. Iniciar un run de MLflow
# El "with" statement asegura que el run se cierre correctamente
with mlflow.start_run():
    # Loguear parámetros
    mlflow.log_param("solver", "saga")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)
    
    # 5. Entrenar el modelo
    model = LogisticRegression(
        solver="saga",
        max_iter=1000,
        penalty="elasticnet",
        l1_ratio=l1_ratio,
        C=alpha, # C es el inverso de la fuerza de regularización
        random_state=42
    )
    model.fit(X_train, y_train)

    # 6. Realizar predicciones y calcular métricas
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    # 7. Loguear métricas
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Verificar URI antes de guardar el modelo
    # print("Artifact URI:", run.info.artifact_uri)
    
    # 8. Loguear el modelo
    # Esto guarda el modelo y sus dependencias para poder cargarlo después
    mlflow.sklearn.log_model(model, "iris_logistic_regression_model")

    print(f"Run MLflow completado. ID del Run: {mlflow.active_run().info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")

2026/03/06 18:03:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run MLflow completado. ID del Run: 5c03958b33b6463995908616c27dd000
Accuracy: 1.0000
F1-Score: 1.0000
🏃 View run blushing-tern-637 at: http://192.168.1.156:5000/#/experiments/9/runs/5c03958b33b6463995908616c27dd000
🧪 View experiment at: http://192.168.1.156:5000/#/experiments/9


In [ ]:
print("Script finalizado.")